# Betting Data Analysis

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data.csv')
print(f'Dataset: {df.shape[0]} bets, {df.shape[1]} columns')
df.head()

Dataset: 1117 bets, 77 columns


,Unique Bet Rule 1,Team,Total?,Unnamed: 3,Bet,Book,Date,Market,Other,LineTaken,...,CZR M,CZR O,BM M$,BM O$,DK M,DK O,4C M,4C M$,4C O,4C O$
0,1,Pistons,0,2024-10,Pistons +30.5 L,FWmb,2024-10-26,-1400,630,-500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Pistons,0,2024-10,Pistons +32.5 L,FWmb,2024-10-26,-1450,640,-480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Pistons,0,2024-10,Pistons +29.5 L,FWmb,2024-10-26,-750,430,-278,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,Clippers,0,2024-10,Clippers ml L,FWmb,2024-10-31,-199,169,-165,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,Clippers,0,2024-10,Clippers ml L,BRmb,2024-10-31,-259,198,-200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# --- Data Cleanup ---

# 1. Strip whitespace from column names
df.columns = df.columns.str.strip()

# 2. Convert Date columns to datetime
df['Date'] = pd.to_datetime(df['Date'])
df['Date.1'] = pd.to_datetime(df['Date.1'])

# 3. Drop Time column
df.drop(columns=['Time'], inplace=True)

# 4. Rename Unnamed: 3 to Month
df.rename(columns={'Unnamed: 3': 'Month'}, inplace=True)

# 5. Rename Unnamed: 16 to BetType
df.rename(columns={'Unnamed: 16': 'BetType'}, inplace=True)

# 6. Total? to bool
df['Total?'] = df['Total?'].astype(bool)

# 7. Grade to categorical
df['Grade'] = df['Grade'].astype('category')

# 8. Drop Ignore columns
df.drop(columns=['Ignore', 'Ignore.1'], inplace=True)

# 9. Drop 100% null columns
null_cols = [c for c in df.columns if df[c].isna().all()]
print(f'Dropping {len(null_cols)} fully null columns: {null_cols}')
df.drop(columns=null_cols, inplace=True)

print(f'\nCleaned: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nDtypes:\n{df.dtypes.to_string()}')

Dropping 8 fully null columns: ['CloseLine', 'CloseOther', 'CloseEdge', 'Trueline', '4C M', '4C M$', '4C O', '4C O$']

Cleaned: 1117 rows, 66 columns

Dtypes:
Unique Bet Rule 1             int64
Team                         object
Total?                         bool
Month                        object
Bet                          object
Book                         object
Date                 datetime64[ns]
Market                        int64
Other                         int64
LineTaken                     int64
Edge                        float64
TrueL                       float64
Decim                       float64
Betsize                     float64
Sport                        object
BetType                      object
Opp                          object
Total Stake                 float64
Rich Stake                  float64
Other Stake                 float64
Grade                      category
Net                         float64
ExpWinPlace                 float64
Date.1       

In [13]:
# --- Analysis DataFrame ---

# Filter to Basketball only
bdf = df[df['Sport'] == 'Basketball'].copy()
print(f'Filtered to Basketball: {len(bdf)} rows')

# Select and rename columns
bdf = bdf[['Unique Bet Rule 1', 'Team', 'Total?', 'Bet', 'Date', 'Market',
            'Other', 'LineTaken', 'Edge', 'BetType', 'Rich Stake', 'Grade']].copy()

bdf.rename(columns={
    'Unique Bet Rule 1': 'Bet',
    'Bet': 'ExactBet',
    'Total?': 'Total',
    'Rich Stake': 'RichStake',
    'LineTaken': 'LineTaken',
}, inplace=True)

# Convert Edge from decimal to percentage with 1 decimal place
bdf['Edge'] = (bdf['Edge'] * 100).round(1)

# Round RichStake to 2 decimal places to fix floating point precision
bdf['RichStake'] = bdf['RichStake'].round(2)

# Calculate Profit
def calc_profit(row):
    if row['Grade'] == 'P':
        return 0.0
    if row['Grade'] == 'L':
        return -row['RichStake']
    # Grade == W: calculate winnings from american odds
    odds = row['LineTaken']
    stake = row['RichStake']
    if odds > 0:
        return stake * (odds / 100)
    else:
        return stake * (100 / abs(odds))

bdf['Profit'] = bdf.apply(calc_profit, axis=1).astype(int)

bdf.to_csv('clean_data.csv', index=False)
print(f'{len(bdf)} bets | Total Profit: ${bdf["Profit"].sum():,}')
print('Exported to clean_data.csv')
bdf.head(10)

Filtered to Basketball: 571 rows
571 bets | Total Profit: $18,588
Exported to clean_data.csv


,Bet,Team,Total,ExactBet,Date,Market,Other,LineTaken,Edge,BetType,RichStake,Grade,Profit
534,221,Purdue,False,Purdue ml L,2025-12-06,112,-142,133,3.8,moneyline,525.0,L,-525
545,227,Utah,False,Utah Jazz ml L,2025-12-08,1953,-11372,15000,607.2,moneyline,20.0,L,-20
548,230,Denver,True,Denver Nuggets u253.5 L,2025-12-11,103,128,120,16.4,total,130.0,W,156
549,231,North,True,North Dakota State u149.5 L,2025-12-11,-142,107,104,11.9,total,230.0,W,239
550,232,Butler,False,Butler ml L,2025-12-13,-103,-123,133,11.6,moneyline,882.0,W,1173
551,233,Cal,True,Cal Riverside u157.5 L,2025-12-13,-115,-102,125,15.7,total,112.0,W,140
552,234,Central,False,Central Arkansas +25.5 L,2025-12-13,-118,-112,130,16.4,spread,1769.0,W,2299
553,234,Central,False,Central Arkansas +15.5 L,2025-12-13,-115,-112,120,10.7,spread,684.0,W,820
554,235,Central,True,Central Arkansas u153.5 L,2025-12-13,-110,-122,128,11.3,total,384.0,L,-384
555,236,Chattanooga,True,Chattanooga o164.5 L,2025-12-13,-123,-108,116,11.3,total,606.0,W,702


In [15]:
# --- Merge rows by Bet ID ---

mdf = pd.read_csv('clean_data.csv')

def grade_agg(grades):
    unique = grades.unique()
    if len(unique) == 1:
        return unique[0]
    return 'mix'

merged = mdf.groupby('Bet').agg(
    Team=('Team', 'first'),
    Total=('Total', 'first'),
    ExactBet=('ExactBet', 'first'),
    Date=('Date', 'first'),
    Market=('Market', 'median'),
    Other=('Other', 'median'),
    LineTaken=('LineTaken', 'median'),
    Edge=('Edge', 'median'),
    BetType=('BetType', 'first'),
    RichStake=('RichStake', 'sum'),
    Grade=('Grade', grade_agg),
    Profit=('Profit', 'sum'),
).reset_index()

# Fix floating point precision
merged['Market'] = merged['Market'].astype(int)
merged['Other'] = merged['Other'].astype(int)
merged['LineTaken'] = merged['LineTaken'].astype(int)
merged['Edge'] = merged['Edge'].round(1)
merged['RichStake'] = merged['RichStake'].round(2)

merged.to_csv('merged_data.csv', index=False)
print(f'Merged: {len(mdf)} rows -> {len(merged)} bets')
print(f'Grade breakdown: {merged["Grade"].value_counts().to_dict()}')
print('Exported to merged_data.csv')
merged.head(10)

Merged: 571 rows -> 373 bets
Grade breakdown: {'L': 182, 'W': 164, 'mix': 26, 'P': 1}
Exported to merged_data.csv


,Bet,Team,Total,ExactBet,Date,Market,Other,LineTaken,Edge,BetType,RichStake,Grade,Profit
0,221,Purdue,False,Purdue ml L,2025-12-06,112,-142,133,3.8,moneyline,525.0,L,-525
1,227,Utah,False,Utah Jazz ml L,2025-12-08,1953,-11372,15000,607.2,moneyline,20.0,L,-20
2,230,Denver,True,Denver Nuggets u253.5 L,2025-12-11,103,128,120,16.4,total,130.0,W,156
3,231,North,True,North Dakota State u149.5 L,2025-12-11,-142,107,104,11.9,total,230.0,W,239
4,232,Butler,False,Butler ml L,2025-12-13,-103,-123,133,11.6,moneyline,882.0,W,1173
5,233,Cal,True,Cal Riverside u157.5 L,2025-12-13,-115,-102,125,15.7,total,112.0,W,140
6,234,Central,False,Central Arkansas +25.5 L,2025-12-13,-116,-112,125,13.6,spread,2453.0,W,3119
7,235,Central,True,Central Arkansas u153.5 L,2025-12-13,-110,-122,128,11.3,total,384.0,L,-384
8,236,Chattanooga,True,Chattanooga o164.5 L,2025-12-13,-123,-108,116,11.3,total,606.0,W,702
9,237,Dayton,False,Dayton -36.5 L,2025-12-13,-114,-110,116,8.9,spread,606.0,L,-606
